# Import

In [187]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from tqdm import tqdm


from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load and Scale Dataset

## Load

In [188]:
iris = load_iris()
X = iris.data
y = iris.target

print("Feature shape:", X.shape)
print("Label shape:", y.shape)


Feature shape: (150, 4)
Label shape: (150,)


## Split

In [189]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.25,   # 0.25 of 0.8 = 0.2 of the full dataset
    random_state=42,
    stratify=y_train_val
)

print("Train size:", X_train.shape[0])
print("Validation size:", X_val.shape[0])
print("Test size:", X_test.shape[0])

Train size: 90
Validation size: 30
Test size: 30


## Standardize

In [190]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)


# Dataloader
A DataLoader is used to load the dataset in small batches during training. It can also shuffle the training data, which helps the model learn better. In PyTorch, it is the standard way to iterate over data efficiently and cleanly.

In [191]:
# Convert data to PyTorch tensors

X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val = torch.tensor(y_val, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)


# Create datasets and dataloaders
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

batch_size = 16

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Model

In [192]:
class base_MLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        
        # A simple network with one hidden layer
        self.model = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        return self.model(x)

In [193]:
class advanced_MLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, dropout_rate=0.3):
        super().__init__()

        # A simple network with one hidden layer and dropout
        self.model = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        return self.model(x)


In [194]:
base_model = base_MLP(4, 16, 3)
advanced_model = advanced_MLP(4, 16, 3, 0.3)

In [195]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
base_model_optimizer = optim.Adam(base_model.parameters(), lr=0.01)
advanced_model_optimzer = optim.Adam(advanced_model.parameters(),lr=0.01)

# Training

## Base model

In [196]:
num_epochs = 100

for epoch in range(num_epochs):
    # -----------------------------
    # Training
    # -----------------------------
    base_model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)

    for batch_X, batch_y in progress_bar:
        # Forward pass
        train_outputs = base_model(batch_X)
        train_loss = criterion(train_outputs, batch_y)

        # Backward pass and parameter update
        base_model_optimizer.zero_grad()
        train_loss.backward()
        base_model_optimizer.step()

        # Collect training statistics
        train_loss_sum += train_loss.item() * batch_X.size(0)
        train_predictions = torch.argmax(train_outputs, dim=1)
        train_correct += (train_predictions == batch_y).sum().item()
        train_total += batch_y.size(0)

        progress_bar.set_postfix({
            "batch_loss": f"{train_loss.item():.4f}"
        })

    train_loss_epoch = train_loss_sum / train_total
    train_acc_epoch = train_correct / train_total

    # -----------------------------
    # Validation
    # -----------------------------
    base_model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            val_outputs = base_model(batch_X)
            val_loss = criterion(val_outputs, batch_y)

            val_loss_sum += val_loss.item() * batch_X.size(0)
            val_predictions = torch.argmax(val_outputs, dim=1)
            val_correct += (val_predictions == batch_y).sum().item()
            val_total += batch_y.size(0)

    val_loss_epoch = val_loss_sum / val_total
    val_acc_epoch = val_correct / val_total

    print(
        f"Epoch [{epoch+1}/{num_epochs}] | "
        f"Train Loss: {train_loss_epoch:.4f} | "
        f"Train Acc: {train_acc_epoch:.4f} | "
        f"Val Loss: {val_loss_epoch:.4f} | "
        f"Val Acc: {val_acc_epoch:.4f}"
    )




Epoch [1/100] | Train Loss: 1.1565 | Train Acc: 0.4556 | Val Loss: 1.0056 | Val Acc: 0.4333


Epoch [2/100] | Train Loss: 0.8737 | Train Acc: 0.5222 | Val Loss: 0.7814 | Val Acc: 0.7333


Epoch [3/100] | Train Loss: 0.6813 | Train Acc: 0.8333 | Val Loss: 0.6102 | Val Acc: 0.8000


Epoch [4/100] | Train Loss: 0.5129 | Train Acc: 0.8778 | Val Loss: 0.4826 | Val Acc: 0.8000


Epoch [5/100] | Train Loss: 0.3888 | Train Acc: 0.8889 | Val Loss: 0.4100 | Val Acc: 0.8000


...

Epoch [96/100] | Train Loss: 0.0379 | Train Acc: 0.9778 | Val Loss: 0.2397 | Val Acc: 0.9333


Epoch [97/100] | Train Loss: 0.0370 | Train Acc: 0.9778 | Val Loss: 0.2324 | Val Acc: 0.9000


Epoch [98/100] | Train Loss: 0.0454 | Train Acc: 0.9778 | Val Loss: 0.2343 | Val Acc: 0.9000


Epoch [99/100] | Train Loss: 0.0485 | Train Acc: 0.9778 | Val Loss: 0.2417 | Val Acc: 0.9333

Epoch [100/100] | Train Loss: 0.0419 | Train Acc: 0.9778 | Val Loss: 0.2350 | Val Acc: 0.9333


In [197]:
from sklearn.metrics import classification_report, confusion_matrix

# -----------------------------
# 8. Final evaluation on the test set
# -----------------------------
base_model.eval()

with torch.no_grad():
    test_outputs = base_model(X_test)
    test_predictions = torch.argmax(test_outputs, dim=1)
    test_accuracy = (test_predictions == y_test).float().mean()

print("\nFinal Test Accuracy:", test_accuracy.item())

# Convert tensors to numpy arrays for scikit-learn
y_true = y_test.numpy()
y_pred = test_predictions.numpy()

# Print classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=iris.target_names))

# Print confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))


Final Test Accuracy: 0.9333333373069763

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30

Confusion Matrix:
[[10  0  0]
 [ 0  9  1]
 [ 0  1  9]]


## Model with Dropout and Early stopping

In [198]:

num_epochs = 1000
patience = 100  # Stop if validation accuracy does not improve for 10 epochs


In [199]:

best_val_accuracy = 0.0
best_model_state = {
    key: value.detach().clone()
    for key, value in advanced_model.state_dict().items()
}
epochs_without_improvement = 0

for epoch in range(num_epochs):
    # -----------------------------
    # Training
    # -----------------------------
    advanced_model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    train_progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)

    for batch_X, batch_y in train_progress:
        # Forward pass
        train_outputs = advanced_model(batch_X)
        loss = criterion(train_outputs, batch_y)

        # Backward pass and parameter update
        advanced_model_optimzer.zero_grad()
        loss.backward()
        advanced_model_optimzer.step()

        # Collect training statistics
        train_loss_sum += loss.item() * batch_X.size(0)
        predictions = torch.argmax(train_outputs, dim=1)
        train_correct += (predictions == batch_y).sum().item()
        train_total += batch_y.size(0)

        train_progress.set_postfix({
            "batch_loss": f"{loss.item():.4f}"
        })

    train_loss = train_loss_sum / train_total
    train_accuracy = train_correct / train_total

    # -----------------------------
    # Validation
    # -----------------------------
    advanced_model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            val_outputs = advanced_model(batch_X)
            val_loss = criterion(val_outputs, batch_y)

            val_loss_sum += loss.item() * batch_X.size(0)
            predictions = torch.argmax(val_outputs, dim=1)
            val_correct += (predictions == batch_y).sum().item()
            val_total += batch_y.size(0)

    val_loss = val_loss_sum / val_total
    val_accuracy = val_correct / val_total

    print(
        f"Epoch [{epoch+1}/{num_epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )

    # -----------------------------
    # Early stopping
    # -----------------------------
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_model_state = {
            key: value.detach().clone()
            for key, value in advanced_model.state_dict().items()
        }
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print("\nEarly stopping triggered.")
        break



Epoch [1/1000] | Train Loss: 1.0666 | Train Acc: 0.4111 | Val Loss: 1.0040 | Val Acc: 0.7000


Epoch [2/1000] | Train Loss: 0.8454 | Train Acc: 0.7556 | Val Loss: 0.7361 | Val Acc: 0.8000


Epoch [3/1000] | Train Loss: 0.6804 | Train Acc: 0.7667 | Val Loss: 0.6305 | Val Acc: 0.8000


Epoch [4/1000] | Train Loss: 0.5699 | Train Acc: 0.8111 | Val Loss: 0.4602 | Val Acc: 0.8000


Epoch [5/1000] | Train Loss: 0.4611 | Train Acc: 0.8444 | Val Loss: 0.6343 | Val Acc: 0.8000


...

Epoch [119/1000] | Train Loss: 0.0427 | Train Acc: 0.9889 | Val Loss: 0.0131 | Val Acc: 0.9333


Epoch [120/1000] | Train Loss: 0.0574 | Train Acc: 0.9667 | Val Loss: 0.0985 | Val Acc: 0.9333


Epoch [121/1000] | Train Loss: 0.0447 | Train Acc: 0.9889 | Val Loss: 0.0066 | Val Acc: 0.9333


Epoch [122/1000] | Train Loss: 0.0743 | Train Acc: 0.9778 | Val Loss: 0.0038 | Val Acc: 0.9333


Epoch [123/1000] | Train Loss: 0.0925 | Train Acc: 0.9667 | Val Loss: 0.0592 | Val Acc: 0.9333

Early stopping triggered.


In [ ]:

# Load the best model
advanced_model.load_state_dict(best_model_state)

advanced_model.eval()
test_correct = 0
test_total = 0

all_y_true = []
all_y_pred = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        test_outputs = advanced_model(batch_X)
        test_predictions = torch.argmax(test_outputs, dim=1)

        # Update accuracy statistics
        test_correct += (test_predictions == batch_y).sum().item()
        test_total += batch_y.size(0)

        # Store predictions and true labels
        all_y_true.extend(batch_y.numpy())
        all_y_pred.extend(test_predictions.numpy())

test_accuracy = test_correct / test_total
print("\nFinal Test Accuracy:", test_accuracy)

# Print classification report
print("\nClassification Report:")
print(classification_report(all_y_true, all_y_pred, target_names=iris.target_names))

# Print confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(all_y_true, all_y_pred))


Final Test Accuracy: 0.9333333333333333

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30

Confusion Matrix:
[[10  0  0]
 [ 0  9  1]
 [ 0  1  9]]
